# Frequency probing: pure tones

## tl;dr

This notebook measures how efficiently seven frequency-band labels can be read
at ten stages of the network: the four encoder blocks, the four decoder blocks,
the input of the output projection and the output quantile head. Every value is
accompanied by a shuffled-label control and by the same reading on an untrained
network. A lock/control reading then compares, stage by stage, how often a tone
on a candidate site and a tone beside it are located. It produces the retrained
16–16 SV figure and the tables for Appendix F. Run from top to bottom to populate
the results; the tables and figures below come from the returned, validated
measurements.

## Context & Methods

Frozen Chronos-Bolt-Tiny checkpoints are read at ten probe points: the [REG]
vector after each of the four encoder blocks (its last position) and after each
of the four decoder blocks (the single query token), the input of the output
projection, and its output, the quantile head. The seven hierarchical binary
frequency tasks are ordered by decision threshold: LL, L, LH, Mid, HL, H and HH.
We retain their existing bands and boundaries.

**Description length by stage.** For each task, unique frequencies are assigned
to five class-stratified prequential blocks. All phases at a frequency remain in
the same block. The first block is charged uniformly at one bit per label; every
later block is encoded by a linear probe trained only on preceding blocks. The
scaler and PCA (up to 20 components) are fitted inside that preceding-data fit,
followed by logistic regression. A fixed seed of 42 governs splits, probe
randomness and both controls. The reported space saving is
\[
SV = 1 - \frac{L_{\mathrm{MDL}}}{n}.
\]
Positive SV means label compression, zero means no saving, and negative values
are retained. The initial uniform block sets a design-dependent upper bound
near 0.8. SV measures the readability of frequency labels from these
representations; it does not measure compression of waveform samples.

**Two controls.** *Shuffled labels*: the same blocks and probe on permuted
labels. *Untrained network*: the same architecture with randomly initialised
weights reads the same contexts and is scored in the same way. What the
untrained network already exposes comes from the tokenisation and the
architecture; only the gap between the two networks can be attributed to training.

**Lock/control reading.** This mirrors the recovery count of the frequency
analysis, read inside the network instead of on the forecast. A tone on a
candidate site (lock) and a tone beside it (control, more than 2 Hz from every
candidate of the geometry) are presented at every phase. A tone is *located* at
a stage when the three band probes on its path (Mid, then L or H, then one of
LL, LH, HL, HH) all return its label, which places it in an eighth of the band.
The probes are fitted on the sweep without the frequencies within 1 Hz of the
lock or control tones, so no located tone was seen in fitting. The outcome is
the located rate at lock and control and their difference, stage by stage. Task
boundaries lie beside several candidate sites, so the reading is repeated on the
pairs whose tones all lie at least 4 Hz from the boundaries on their path.

### Key assumptions

- Signals are sampled at 512 Hz. The full grid is 2–250 Hz in 1-Hz steps,
  with ten explicitly distinct phases per frequency. Candidate sites keep
  their exact, possibly non-integer, value in the lock/control reading.
- Context length is the largest \(L\le512\) satisfying \(L\bmod P=0\) and
  \((L-P)\bmod S=0\), avoiding masked patch padding and uncovered samples.
- The published checkpoint and the fifteen retrained geometries use their
  existing, identified weights; no model is retrained here.
- The grouped five-block estimator is the repository's adapted protocol.
  The figure layout follows [Pagani et al., Figure 3](https://arxiv.org/html/2605.06361v1).
- These values are not those of `02_mdl_bandtasks.parquet` (Model B), whose
  blocks are not grouped by frequency.

### Set up the experiment

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

REPO = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "support_scripts" / "probing_lib.py").is_file()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook from the patchAliasing repository.")
sys.path.insert(0, str(REPO / "notebooks" / "support_scripts"))

from probing_protocol import ProbeConfig, context_length, make_design
from probing_lib import MODELS

MODE = os.environ.get("PATCHALIASING_PROBING_MODE", "full").strip().lower()
if MODE not in {"full", "smoke"}:
    raise ValueError("PATCHALIASING_PROBING_MODE must be 'full' or 'smoke'.")
cfg = ProbeConfig.smoke() if MODE == "smoke" else ProbeConfig()
models = tuple(model for model in MODELS if model.tag == "retrained-p16-s16") if MODE == "smoke" else MODELS
ROOT = REPO / "notebooks" / "_run" / "probing"
DEVICE = "cpu"
TASK_ORDER = ["LL", "L", "LH", "Mid", "HL", "H", "HH"]
STAGE_ORDER = ["enc_0", "enc_1", "enc_2", "enc_3", "dec_0", "dec_1", "dec_2", "dec_3",
               "output_reg", "output_head"]
# Tracked copies of the values the report quotes (notebooks/_run is not in the repository).
PUBLISHED = REPO / "notebooks" / "signal_analysis" / "results" / "probing"

print(f"Mode: {MODE}; device: {DEVICE}; checkpoints: {len(models)}")
if MODE == "smoke":
    display(Markdown("**SMOKE CHECK — reduced design; these results are not report estimates.**"))

def lock_control_panel(summary, family, network="trained", suffix=""):
    """Geometry by stage: located rate at lock / at control, in per cent."""
    part = summary.loc[summary["family"].eq(family) & summary["network"].eq(network)].copy()
    part["cell"] = [("-" if pd.isna(l) or pd.isna(c) else f"{l:.0f}/{c:.0f}")
                    for l, c in zip(part[f"lock_pct{suffix}"], part[f"control_pct{suffix}"])]
    return part.pivot(index="model", columns="stage", values="cell").reindex(
        index=[model.tag for model in models], columns=STAGE_ORDER)


def net_panel(summary, family, network="trained", suffix=""):
    part = summary.loc[summary["family"].eq(family) & summary["network"].eq(network)]
    return part.pivot(index="model", columns="stage", values=f"net_pct{suffix}").reindex(
        index=[model.tag for model in models], columns=STAGE_ORDER).round(1)


def publish(paths):
    """Copy full-design exports next to the notebook so the report values are traceable."""
    import shutil
    if MODE != "full":
        return []
    PUBLISHED.mkdir(parents=True, exist_ok=True)
    copied = []
    for path in paths:
        shutil.copy2(path, PUBLISHED / path.name)
        copied.append(PUBLISHED / path.name)
    return copied

## Data

Every example is an isolated unit-amplitude sinusoid. The ten full-design
phases are explicitly distinct over one cycle, replacing the legacy helper
that can repeat phases at some integer frequencies. All checkpoints receive
the same frequency/phase design, truncated only by their exact context rule.

In [ ]:
design = make_design(cfg)
coverage = pd.DataFrame([
    {"model": model.tag, "P": model.P, "S": model.S,
     "context_samples": context_length(model.P, model.S)}
    for model in models
])
display(coverage)
print(f"{design['frequency'].nunique()} frequencies × {cfg.n_phases} phases = {len(design)} contexts per sweep")
display(design.head(6))

## Results

### Run or resume the sweep

The runner processes one checkpoint at a time, records checkpoint and package
identities, and reuses only shards matching the exact configuration, input
hashes and code. A failure stays visible; the notebook does not substitute
partial results. The full output has 16 × 10 × 7 = 1,120 task/stage rows, each
with its two controls, and one lock/control row per tone, stage and network.

In [ ]:
from probing_lib import run_pure

results, run_dir = run_pure(ROOT, cfg=cfg, models=models, device=DEVICE)
trials = pd.read_csv(run_dir / "lock_control.csv")
prefix = "smoke_" if MODE == "smoke" else ""
lock_control = pd.read_csv(run_dir / f"{prefix}pure_lock_control.csv")
print(f"Validated SV rows: {len(results):,}; lock/control rows: {len(trials):,}")
print(f"Run directory: {run_dir}")

### Inspect the retrained 16–16

In [ ]:
reference = results.loc[results["model"].eq("retrained-p16-s16")].copy()
if reference.empty:
    raise ValueError("The reference checkpoint is missing from the completed run.")
for column, name in (("SV", "trained network"), ("SV_shuffled", "shuffled-label control"),
                     ("SV_random_init", "untrained-network control")):
    display(reference.pivot(index="stage", columns="task", values=column)
            .reindex(index=STAGE_ORDER, columns=TASK_ORDER).round(3).rename_axis(name))

### Figure for Section 5.1

Seven bars within each stage show true-task SV; the parenthesised labels give
the shuffled-label and untrained-network controls. The zero line remains
visible and negative values are preserved. The image carries no title or note:
they belong to the caption. PNG and PDF exports share the same computed data.

In [ ]:
figure_paths = sorted(run_dir.glob(f"{prefix}fig_probing_sv_p16_s16.png"))
if len(figure_paths) != 1:
    raise RuntimeError(f"Expected one reference figure, found {len(figure_paths)}.")
display(Image(filename=str(figure_paths[0]), width=900))
print(figure_paths[0])

### All geometries, stage by stage

Median SV over the seven tasks at each stage, for the trained network and for
the untrained one. A trained value well above its untrained counterpart is
information the training added; comparable values are what the architecture
and the tokenisation already expose.

In [ ]:
medians = results.groupby(["model", "stage"], sort=False)[["SV", "SV_random_init", "SV_shuffled"]].median()
order = [model.tag for model in models]
for column, name in (("SV", "trained network"), ("SV_random_init", "untrained network")):
    display(medians[column].unstack("stage").reindex(index=order, columns=STAGE_ORDER).round(2).rename_axis(name))
gap = (medians["SV"] - medians["SV_random_init"]).unstack("stage").reindex(index=order, columns=STAGE_ORDER)
display(gap.round(2).rename_axis("trained minus untrained"))
print(f"Shuffled-label control, largest median over all cells: {medians['SV_shuffled'].max():+.3f}")

### Lock/control reading

Each cell is the percentage of contexts located with the tone on a candidate
site / with the tone beside it. The second table is their difference in
percentage points: negative means the band of a tone is harder to read on a
site than beside it. The third repeats the difference on the pairs that lie
away from the task boundaries; `pairs_off` says how many pairs that leaves, and
a geometry with few of them cannot separate a site from a boundary. The last
table is the same difference on the untrained network.

In [ ]:
display(lock_control_panel(lock_control, "pure").rename_axis("lock/control located, %"))
display(net_panel(lock_control, "pure").rename_axis("net (lock - control), points"))
display(net_panel(lock_control, "pure", suffix="_off").rename_axis("net, off-boundary pairs"))
display(lock_control.loc[lock_control["network"].eq("trained") & lock_control["stage"].eq("output_head"),
                         ["model", "pairs", "pairs_off"]].set_index("model").reindex(order))
display(net_panel(lock_control, "pure", network="random_init").rename_axis("net, untrained network"))

In [ ]:
# The same reading by branch of the candidate set, pooled over geometries (trained network).
by_branch = trials.loc[trials["network"].eq("trained")].groupby(["branch", "stage", "arm"]).agg(
    located=("located", "sum"), contexts=("n_contexts", "sum"))
by_branch["pct"] = 100 * by_branch["located"] / by_branch["contexts"]
display(by_branch["pct"].unstack("arm")[["lock", "control"]].unstack("stage")
        .reindex(columns=STAGE_ORDER, level="stage").round(0))

### Export inventory

In [ ]:
exported = sorted(path for path in run_dir.rglob("*") if path.is_file() and path.suffix in {".png", ".pdf", ".tex", ".txt"})
print(f"{len(exported)} figure/table/caption export files; raw rows are also saved as CSV.")
display(pd.DataFrame({"export": [str(path.relative_to(run_dir)) for path in exported]}).head(24))
copied = publish([path for path in run_dir.iterdir() if path.is_file() and (
    path.name.startswith(("pure_", "fig_probing_sv_p16_s16", "probing_")) and path.suffix in {".csv", ".tex", ".pdf", ".png", ".txt"})])
print(f"{len(copied)} files copied to {PUBLISHED}" if copied else "Smoke mode: nothing copied to the tracked folder.")

## Takeaways

The summary below is computed from the seven reference-checkpoint tasks.
Compare true-task values with both controls before interpreting positive SV as
readable frequency structure, and read the lock/control differences next to
the number of off-boundary pairs. These are descriptive probes; they do not
decide the Bayesian hypotheses or establish causal relevance.

In [ ]:
summary = reference.groupby("stage", sort=False).agg(
    tasks=("task", "size"),
    positive_sv_tasks=("SV", lambda values: int((values > 0).sum())),
    min_sv=("SV", "min"), median_sv=("SV", "median"), max_sv=("SV", "max"),
    median_shuffled_sv=("SV_shuffled", "median"),
    max_shuffled_sv=("SV_shuffled", "max"),
    median_random_init_sv=("SV_random_init", "median"),
).reindex(STAGE_ORDER)
display(summary.round(3))
print(f"SV ceiling by task (1 - first block / n): "
      f"{(1 - reference.first_chunk_n / reference.n).min():.2f} to {(1 - reference.first_chunk_n / reference.n).max():.2f}")
if MODE == "smoke":
    display(Markdown("**Validation result only:** rerun in full mode before using the figure or the tables in the report."))
else:
    display(Markdown("The full-design exports are ready for Section 5.1 and Appendix F."))